[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/jvm-internals/blob/main/notebooks/playgrounds/classfile.ipynb)

**Can you write a class file by hand, one field at a time, and have javap and the JVM agree with you?**

Run the cell below first. Everything after it assumes `jvx` is loaded, and the numbers on this page came from `jdk-27+35`.

The class file you are about to write is 299 bytes of nothing mysterious. Every one of them is written down in one chapter of one document, in the order they appear, and you can type them.

Almost nobody does. The tools in between are too good: `javac` produces a class file from Java and shows you none of it, and `java.lang.classfile` produces one from a builder and hides the constant pool, the offsets and the lengths, which is what you want when writing a tool and the opposite of what you want now.

So this page has no builder. You write the fields, in the order the specification lists them, and three parties tell you what they made of it: `javap`, the VM in this kernel, and a fresh JVM that runs them.

Nothing here is a quiz. Change a number, run the cell again, and see who complains.

In [ ]:
// Generated by tools/build.py. Do not edit this cell, it is overwritten on
// every build. Edit the sources and run `python tools/build.py notebooks`.
//
// helper surface   jvx/00-imports.jsh, jvx/05-ui.jsh, jvx/10-markword.jsh, jvx/12-classfile.jsh, jvx/15-gate.jsh, jvx/18-lens.jsh, jvx/20-jvx.jsh
// bit positions    docs/generated/markword.json, read by tools/gen_markword.py
//                  from src/hotspot/share/oops/markWord.hpp at jdk-27+35
//
// This cell assumes a Java kernel is already running. Installing the pinned
// JDK from a cold Colab runtime is a separate step that is still being
// measured, and it goes here when it is. See issue #1.

// JShell imports a useful default set, but not these. They are separate snippets on
// purpose: an import in JShell applies to everything typed afterwards, so putting
// them first means a reader's own cells get them too without asking.
//
// The second group is here for a reason worth knowing about, because it cost an
// afternoon. A jshell you start in a terminal imports java.nio.file.* for you. The
// notebook kernel does not: JJava sets its own list, which is java.util, java.io,
// java.math, java.net, java.time, java.util.concurrent, java.util.prefs and
// java.util.regex, and nothing else. So a helper surface that loads perfectly in a
// terminal can fail to compile in the kernel every reader uses, and the error a reader
// sees is `cannot find symbol: variable jvx`, which points at the wrong thing entirely.
// tools/test_jvx_ui.py loads the whole surface with only the kernel's imports for that
// reason.
import java.lang.management.ManagementFactory;
import java.lang.management.RuntimeMXBean;
import java.lang.reflect.Field;
import java.lang.reflect.Method;
import java.lang.reflect.Modifier;
import com.sun.management.HotSpotDiagnosticMXBean;
import com.sun.management.VMOption;

import java.nio.charset.StandardCharsets;
import java.nio.file.Files;
import java.nio.file.Path;

// The third group is the class file playground's, and it is the JDK's own class file
// library rather than anything this project wrote. Every opcode number, constant pool
// tag and access flag bit a reader meets comes from here, read off the JDK they are
// running, so the playground cannot disagree with the platform it is teaching.
import java.lang.classfile.ClassFile;
import java.lang.classfile.Opcode;
import java.lang.classfile.constantpool.PoolEntry;
import java.util.spi.ToolProvider;

// Everything a lesson draws on the screen goes through here, and the shape of it comes
// straight out of a measurement rather than out of taste.
//
// probes/widgets measured twelve ways of getting something in front of a reader, in four
// places each, and a saved notebook that nobody has run keeps four of them: a style
// attribute on the element, details and summary, an img whose src is an SVG data URI, and
// markdown. Everything else is sanitized away. Style tags are removed while the class
// attribute is kept, so the rule is gone and the hook that wanted it is still there. An id
// is renamed to data-jupyter-id, so every selector quietly stops matching. Form controls
// arrive disabled. Scripts, onclick and iframes are removed outright.
//
// That is the state a reader is in when they click a link and read the page, which is most
// readers most of the time. So this file uses those four things and nothing else. There is
// no style tag, no id, no script and no input anywhere below, and tools/test_jvx_ui.py
// fails the build if one appears. The full reading is in docs/probes/widgets.md.

class Ui {

    static final String FONT = "system-ui, -apple-system, Segoe UI, Roboto, sans-serif";
    static final String MONO = "ui-monospace, SFMono-Regular, Menlo, Consolas, monospace";
    static final String INK = "#212529";
    static final String MUTED = "#868e96";
    static final String BLUE = "#4c6ef5";
    static final String GREEN = "#2f9e44";
    static final String ORANGE = "#e8590c";

    // -- getting a payload to the front end ----------------------------------------
    //
    // The kernel is JJava, and the `display` it puts in scope is a static method on
    // org.dflib.jjava.jupyter.kernel.BaseNotebookStatics. Calling it by name would work
    // in a notebook and would break everywhere else, because the same helper surface
    // gets piped into a plain jshell when somebody debugs the bootstrap, and JShell
    // refuses to run a method whose body names something that does not exist. Looking it
    // up reflectively answers both questions at once: whether there is a screen to draw
    // on, and how to draw on it.

    private static Method displayMethod;
    private static boolean lookedForIt = false;

    /** Is there a front end here that can render markup, or are we in a terminal. */
    static boolean rich() {
        if (!lookedForIt) {
            lookedForIt = true;
            try {
                Class<?> statics =
                    Class.forName("org.dflib.jjava.jupyter.kernel.BaseNotebookStatics");
                displayMethod = statics.getMethod("display", Object.class, String[].class);
            } catch (Throwable notANotebook) {
                displayMethod = null;
            }
        }
        return displayMethod != null;
    }

    /**
     * Put markup on the screen. False means there was no screen, so print text instead.
     *
     * The return value of `display` is thrown away on purpose and this is the only place
     * in the project that calls it. `display` hands back the id it assigned, JShell
     * prints the value of the last expression it evaluates, and the result is a line of
     * hex under every widget on the page. Measured on twelve cells out of twelve. One
     * assignment here is the whole fix.
     */
    static boolean html(String markup) {
        if (!rich()) return false;
        try {
            String ignored = (String) displayMethod.invoke(
                null, markup, new String[] { "text/html" });
            return true;
        } catch (Throwable t) {
            // A front end that turned out not to want it is not worth an exception in a
            // reader's face. Saying false sends the caller to the text version.
            return false;
        }
    }

    // -- building the markup ---------------------------------------------------------

    static String esc(String text) {
        return text.replace("&", "&amp;")
                   .replace("<", "&lt;")
                   .replace(">", "&gt;")
                   .replace("\"", "&quot;");
    }

    /** Escaped text, with `backticks` turned into code spans, which lessons already write. */
    static String prose(String text) {
        String[] parts = esc(text).split("`", -1);
        StringBuilder out = new StringBuilder();
        for (int i = 0; i < parts.length; i++) {
            // The odd numbered pieces are the ones between a pair of backticks. An
            // unclosed backtick leaves the last piece odd, and that one gets its backtick
            // put back and goes out as plain text, because a lesson with a typo in it
            // should look wrong on the page rather than quietly lose a character.
            if (i % 2 == 1 && i < parts.length - 1) {
                out.append(code(parts[i]));
            } else if (i % 2 == 1) {
                out.append("`").append(parts[i]);
            } else {
                out.append(parts[i]);
            }
        }
        return out.toString();
    }

    /** Already escaped text in a code span. Use prose() for anything a lesson typed. */
    static String code(String escaped) {
        return "<code style=\"font-family:" + MONO + ";font-size:0.92em;background:#e9ecef;"
            + "padding:1px 5px;border-radius:3px\">" + escaped + "</code>";
    }

    static String card(String accent, String label, String body) {
        return "<div style=\"font-family:" + FONT + ";color:" + INK + ";max-width:46em;"
            + "border:1px solid #dee2e6;border-left:5px solid " + accent + ";"
            + "border-radius:6px;background:#f8f9fa;padding:14px 16px;margin:4px 0\">"
            + "<div style=\"font-size:11px;font-weight:700;letter-spacing:0.08em;"
            + "text-transform:uppercase;color:" + accent + ";margin-bottom:8px\">"
            + esc(label) + "</div>"
            + body
            + "</div>";
    }

    /**
     * The one interactive element that survives everywhere.
     *
     * No CSS and no JavaScript, so there is nothing for a sanitizer to take away. Open it
     * when the reader has earned what is inside and leave it shut when they have not.
     */
    static String details(String summary, String body, boolean open) {
        return "<details" + (open ? " open" : "") + " style=\"margin-top:10px\">"
            + "<summary style=\"cursor:pointer;font-weight:600;color:" + BLUE + "\">"
            + esc(summary) + "</summary>"
            + "<div style=\"margin-top:8px\">" + body + "</div></details>";
    }

    /**
     * A picture, as an img with the SVG base64 encoded into the src.
     *
     * This is the useful half of the whole widget probe. An `image/svg+xml` output is
     * shown as escaped source text in a notebook nobody has run, which is worse than
     * showing nothing, and the identical bytes inside an img render in every environment
     * measured. So anything this project can draw as an SVG, it can put on any page.
     */
    static String img(String svg, String alt) {
        String encoded = Base64.getEncoder().encodeToString(svg.getBytes(StandardCharsets.UTF_8));
        return "<img alt=\"" + esc(alt) + "\" style=\"max-width:100%;display:block;"
            + "margin:4px 0\" src=\"data:image/svg+xml;base64," + encoded + "\">";
    }

    static String line(String body) {
        return "<div style=\"margin:4px 0;line-height:1.5\">" + body + "</div>";
    }

    static String small(String body) {
        return "<div style=\"margin-top:10px;font-size:13px;color:" + MUTED
            + ";line-height:1.5\">" + body + "</div>";
    }
}

// The mark word is the first eight bytes of every object on the heap, and it carries
// several unrelated things at once: the lock state, the identity hash, the age the
// collector uses to decide about promotion, and on JDK 27 the class pointer as well.
//
// None of the numbers below are typed by hand. The placeholder line further down is
// replaced by tools/build.py using docs/generated/markword.json, which
// tools/gen_markword.py works out from HotSpot's own markWord.hpp at the pinned tag.
// The citation on each field is the line of markWord.hpp it came from, so a reader
// who does not believe a number can go and look at the line that produced it.

class MarkWord {

    static final class Field {
        final String name;
        final int shift;
        final int bits;
        final String meaning;
        final String citation;

        Field(String name, int shift, int bits, String meaning, String citation) {
            this.name = name;
            this.shift = shift;
            this.bits = bits;
            this.meaning = meaning;
            this.citation = citation;
        }

        /** The value of this field in the given word. */
        long of(long word) {
            return (word >>> shift) & ((1L << bits) - 1);
        }

        /** The bit positions this field occupies, high end first, the way a diagram reads. */
        String span() {
            return (shift + bits - 1) + ".." + shift;
        }
    }

    static final String SOURCE_PATH = "src/hotspot/share/oops/markWord.hpp";
    static final String SOURCE_TAG = "jdk-27+35";
    static final String SOURCE_SHA256 = "c249a091cf7dcae455d073a32f29d65b0bf2c1f0b452bc5705b706a6f1aba267";

    static final Field[] FIELDS = {
        new Field("lock", 0, 2,
            "the lock state, and whether a collector has marked or forwarded the object",
            "src/hotspot/share/oops/markWord.hpp:124@jdk-27+35"),
        new Field("self_fwd", 2, 1,
            "set when a collector forwarded the object in place",
            "src/hotspot/share/oops/markWord.hpp:125@jdk-27+35"),
        new Field("age", 3, 4,
            "how many collections the object has survived",
            "src/hotspot/share/oops/markWord.hpp:126@jdk-27+35"),
        new Field("valhalla", 7, 4,
            "reserved for Valhalla, unused today",
            "src/hotspot/share/oops/markWord.hpp:127@jdk-27+35"),
        new Field("hash", 11, 31,
            "the identity hash, written the first time anything asks for it",
            "src/hotspot/share/oops/markWord.hpp:128@jdk-27+35"),
        new Field("klass", 42, 22,
            "the compressed class pointer, present only when UseCompactObjectHeaders is on",
            "src/hotspot/share/oops/markWord.hpp:150@jdk-27+35"),
    };

    static final String[] LOCK_BITS = {"00", "01", "10", "11"};
    static final String[] LOCK_MEANING = {
        "locked, a stack lock is held, and the real header is in the lock record",
        "unlocked, the ordinary state",
        "monitor, the lock is inflated",
        "marked, a collector is using the word, and the real header is elsewhere",
    };

    static Field field(String name) {
        for (Field f : FIELDS) {
            if (f.name.equals(name)) return f;
        }
        throw new IllegalArgumentException(
            "no field called " + name + " in the mark word at " + SOURCE_TAG);
    }

    static long get(long word, String name) {
        return field(name).of(word);
    }

    static String hex(long word) {
        return String.format("0x%016x", word);
    }

    /** All 64 bits, grouped in bytes, so the reader can count them. */
    static String bin(long word) {
        StringBuilder b = new StringBuilder(72);
        for (int i = 63; i >= 0; i--) {
            b.append((word >>> i) & 1L);
            if (i % 8 == 0 && i != 0) b.append(' ');
        }
        return b.toString();
    }

    /**
     * The same 64 bits with each field's own bits shown and everything else dimmed to
     * a dot. Reading a hex number and believing where the boundaries are is exactly
     * the step where people go wrong, so this draws the boundaries.
     */
    static String ruler(long word) {
        StringBuilder out = new StringBuilder();
        for (int i = FIELDS.length - 1; i >= 0; i--) {
            Field f = FIELDS[i];
            StringBuilder row = new StringBuilder(72);
            for (int bit = 63; bit >= 0; bit--) {
                boolean mine = bit >= f.shift && bit < f.shift + f.bits;
                row.append(mine ? Character.forDigit((int) ((word >>> bit) & 1L), 10) : '.');
                if (bit % 8 == 0 && bit != 0) row.append(' ');
            }
            out.append(row).append("  ").append(f.name).append('\n');
        }
        return out.toString();
    }

    static String decode(long word) {
        StringBuilder b = new StringBuilder();
        b.append(hex(word)).append('\n');
        b.append(bin(word)).append('\n');
        b.append('\n');
        b.append(ruler(word));
        b.append('\n');
        b.append(String.format("%-8s %-9s %-12s %s%n", "bits", "field", "value", "meaning"));
        b.append("-".repeat(78)).append('\n');
        for (int i = FIELDS.length - 1; i >= 0; i--) {
            Field f = FIELDS[i];
            b.append(String.format("%-8s %-9s %-12s %s%n",
                f.span(), f.name, "0x" + Long.toHexString(f.of(word)), f.meaning));
        }
        b.append('\n');
        int lock = (int) get(word, "lock");
        b.append("lock ").append(LOCK_BITS[lock]).append(", ").append(LOCK_MEANING[lock]).append('\n');
        return b.toString();
    }

    /** Where every number above came from, for a reader who wants to check one. */
    static String provenance() {
        StringBuilder b = new StringBuilder();
        b.append("generated from ").append(SOURCE_PATH).append(" at ").append(SOURCE_TAG).append('\n');
        b.append("sha256 ").append(SOURCE_SHA256).append('\n');
        b.append('\n');
        for (int i = FIELDS.length - 1; i >= 0; i--) {
            Field f = FIELDS[i];
            b.append(String.format("%-9s %s%n", f.name, f.citation));
        }
        return b.toString();
    }
}

// The Class File Playground. A class file you write one field at a time, and three
// independent opinions on what you wrote.
//
// The thing this exists to make possible is a reader typing the bytes of a class file
// into a cell, running it, and watching `javap` and the VM agree with them. Every other
// way of producing a class file hides the part worth seeing. A compiler hides all of it.
// java.lang.classfile hides the constant pool, the offsets and the lengths, which is
// exactly right for writing a tool and exactly wrong for learning what a class file is.
//
// So Cf writes nothing on its own. Every field in the file is a call the reader makes,
// in the order the specification lists them {[JVMS 4.1@SE25]}, and Cf does three things
// that carry no teaching and a great deal of arithmetic:
//
//   1. It hands out constant pool indices, because an index is a number you cannot know
//      until you have decided what else is in the pool, and getting one wrong produces
//      an error message about something else entirely.
//   2. It fills in the length fields, because a length is the size of what comes after
//      it, which you cannot write until you have written the rest.
//   3. It remembers what every byte was for, so the dump can say so.
//
// Not one number below is typed in. The magic number, the access flag bits, the constant
// pool tags and every opcode are read off the JDK the reader is running, through
// java.lang.classfile, which is the same library javac writes class files with. A table
// of tag numbers copied into this file would be a table that is right today and wrong on
// the JDK that adds a constant kind, and it would be the kind of wrong nobody notices.

class Cf {

    /** One run of bytes, and what it was for. The dump is a list of these. */
    record Span(String label, int at, int length, String note) {}

    private byte[] buf = new byte[512];
    private int len = 0;
    private final List<Span> spans = new ArrayList<>();

    // A length field that has been written but not yet filled in, as the offset of the
    // field, its width in bytes, and which span to relabel once the answer is known.
    private final Deque<int[]> pending = new ArrayDeque<>();

    // The constant pool, collected before any of it is written, because everything else
    // in the file points into it and a reader needs the index before they can point.
    private final List<byte[]> entries = new ArrayList<>();
    private final List<String> entryText = new ArrayList<>();
    private final List<Integer> entryIndex = new ArrayList<>();
    private final Map<String, Integer> alreadyThere = new LinkedHashMap<>();
    private int nextIndex = 1;
    private boolean poolWritten = false;

    // -- the constant pool ------------------------------------------------------------
    //
    // Two things about it surprise everybody once, and both are visible here rather than
    // explained. The first index is 1 and not 0 {[JVMS 4.1@SE25]}, so a valid index is
    // never zero and a zero where an index belongs means "no entry", which is why
    // super_class of java/lang/Object is 0 and nothing else in a class file is. And a
    // long or a double takes two slots, so the entry after one is numbered two higher
    // {[JVMS 4.4.5@SE25]}. Ask for a long and then a string, and look at the two numbers
    // that come back.

    private int add(String key, String text, int slots, byte[] body) {
        Integer found = alreadyThere.get(key);
        if (found != null) return found;
        if (poolWritten) {
            throw new IllegalStateException(
                "the constant pool has already been written into the file, so index "
                + nextIndex + " for " + text + " would land after the entries. Ask for "
                + "every index you need before calling pool()");
        }
        int index = nextIndex;
        entries.add(body);
        entryText.add(text);
        entryIndex.add(index);
        alreadyThere.put(key, index);
        nextIndex += slots;
        return index;
    }

    /**
     * Modified UTF-8, which is not UTF-8 and differs in two places {[JVMS 4.4.7@SE25]}.
     *
     * A zero character is two bytes rather than one, so a name can contain a zero and
     * still be scanned by C code looking for a terminator. And a character outside the
     * basic plane is written as its two surrogates, three bytes each, rather than as one
     * four byte sequence. For ASCII the two encodings are identical, which is why using
     * the wrong one works for a long time and then does not.
     */
    static byte[] modifiedUtf8(String text) {
        ByteArrayOutputStream out = new ByteArrayOutputStream();
        for (int i = 0; i < text.length(); i++) {
            char c = text.charAt(i);
            if (c != 0 && c < 0x80) {
                out.write(c);
            } else if (c < 0x800) {
                out.write(0xc0 | (c >> 6));
                out.write(0x80 | (c & 0x3f));
            } else {
                out.write(0xe0 | (c >> 12));
                out.write(0x80 | ((c >> 6) & 0x3f));
                out.write(0x80 | (c & 0x3f));
            }
        }
        return out.toByteArray();
    }

    private static byte[] entry(int tag, byte[] rest) {
        byte[] all = new byte[rest.length + 1];
        all[0] = (byte) tag;
        System.arraycopy(rest, 0, all, 1, rest.length);
        return all;
    }

    private static byte[] u2bytes(int... values) {
        byte[] out = new byte[values.length * 2];
        for (int i = 0; i < values.length; i++) {
            out[i * 2] = (byte) (values[i] >> 8);
            out[i * 2 + 1] = (byte) values[i];
        }
        return out;
    }

    private static byte[] u4bytes(long value) {
        return new byte[] {
            (byte) (value >> 24), (byte) (value >> 16), (byte) (value >> 8), (byte) value };
    }

    /** A string, which is what a name, a descriptor and a literal are all made of. */
    int utf8(String text) {
        byte[] raw = modifiedUtf8(text);
        byte[] body = new byte[raw.length + 2];
        body[0] = (byte) (raw.length >> 8);
        body[1] = (byte) raw.length;
        System.arraycopy(raw, 0, body, 2, raw.length);
        return add("utf8:" + text, "Utf8 " + quoted(text),
            1, entry(PoolEntry.TAG_UTF8, body));
    }

    /** A class, named the way the class file names one: slashes, and no L or semicolon. */
    int classEntry(String internalName) {
        int name = utf8(internalName);
        return add("class:" + internalName, "Class #" + name + " " + internalName,
            1, entry(PoolEntry.TAG_CLASS, u2bytes(name)));
    }

    /** A string literal, which is a pointer to a Utf8 and not the characters themselves. */
    int stringEntry(String text) {
        int value = utf8(text);
        return add("string:" + text, "String #" + value + " " + quoted(text),
            1, entry(PoolEntry.TAG_STRING, u2bytes(value)));
    }

    /** A name and a descriptor together, which is what a member is identified by. */
    int nameAndType(String name, String descriptor) {
        int n = utf8(name);
        int d = utf8(descriptor);
        return add("nat:" + name + " " + descriptor,
            "NameAndType #" + n + ":#" + d + " " + name + " " + descriptor,
            1, entry(PoolEntry.TAG_NAME_AND_TYPE, u2bytes(n, d)));
    }

    private int member(int tag, String kind, String owner, String name, String descriptor) {
        int c = classEntry(owner);
        int nat = nameAndType(name, descriptor);
        return add(kind + ":" + owner + "." + name + descriptor,
            kind + " #" + c + ".#" + nat + " " + owner + "." + name + ":" + descriptor,
            1, entry(tag, u2bytes(c, nat)));
    }

    int fieldref(String owner, String name, String descriptor) {
        return member(PoolEntry.TAG_FIELDREF, "Fieldref", owner, name, descriptor);
    }

    int methodref(String owner, String name, String descriptor) {
        return member(PoolEntry.TAG_METHODREF, "Methodref", owner, name, descriptor);
    }

    int interfaceMethodref(String owner, String name, String descriptor) {
        return member(PoolEntry.TAG_INTERFACE_METHODREF, "InterfaceMethodref",
            owner, name, descriptor);
    }

    int integerEntry(int value) {
        return add("int:" + value, "Integer " + value,
            1, entry(PoolEntry.TAG_INTEGER, u4bytes(value)));
    }

    /** Two slots, not one, and the index after this one proves it. */
    int longEntry(long value) {
        byte[] body = new byte[8];
        for (int i = 0; i < 8; i++) body[i] = (byte) (value >> (56 - i * 8));
        return add("long:" + value, "Long " + value + " (takes two slots)",
            2, entry(PoolEntry.TAG_LONG, body));
    }

    private static String quoted(String text) {
        return "\"" + text.replace("\n", "\\n") + "\"";
    }

    // -- writing the file --------------------------------------------------------------

    private void ensure(int more) {
        if (len + more <= buf.length) return;
        byte[] bigger = new byte[Math.max(buf.length * 2, len + more)];
        System.arraycopy(buf, 0, bigger, 0, len);
        buf = bigger;
    }

    private Cf put(String label, int width, long value, String note) {
        ensure(width);
        int at = len;
        for (int i = width - 1; i >= 0; i--) buf[len++] = (byte) (value >> (i * 8));
        spans.add(new Span(label, at, width, note));
        return this;
    }

    /** One byte. */
    Cf u1(String label, int value) { return put(label, 1, value, String.valueOf(value)); }

    /** Two bytes, high one first, which is how every multi byte field in a class file is. */
    Cf u2(String label, int value) { return put(label, 2, value, String.valueOf(value)); }

    Cf u2(String label, int value, String note) { return put(label, 2, value, note); }

    Cf u4(String label, long value) { return put(label, 4, value, String.valueOf(value)); }

    Cf u4(String label, long value, String note) { return put(label, 4, value, note); }

    /** Bytes you have already got, appended as they are. */
    Cf raw(String label, byte[] value) {
        ensure(value.length);
        int at = len;
        System.arraycopy(value, 0, buf, len, value.length);
        len += value.length;
        spans.add(new Span(label, at, value.length, value.length + " bytes"));
        return this;
    }

    /**
     * One instruction, named rather than numbered.
     *
     * The number comes from java.lang.classfile.Opcode on the JDK this is running on,
     * so the byte a reader gets is the byte that JDK's own class file writer would emit
     * for that instruction, and nothing in this project chose it.
     */
    Cf op(String mnemonic) {
        int code = opcode(mnemonic);
        return put(mnemonic, 1, code, "opcode 0x" + Integer.toHexString(code));
    }

    /** The opcode byte for an instruction name, read off the JDK. */
    static int opcode(String mnemonic) {
        try {
            return Opcode.valueOf(mnemonic.toUpperCase(Locale.ROOT)).bytecode();
        } catch (IllegalArgumentException e) {
            throw new IllegalArgumentException(
                "this JDK's java.lang.classfile.Opcode has no instruction called "
                + mnemonic + ". The names are the ones in docs/generated/opcodes.md");
        }
    }

    /** What instruction a byte is, or null when the specification assigns it nothing. */
    static String mnemonic(int code) {
        for (Opcode o : Opcode.values()) {
            if (o.bytecode() == (code & 0xff) && !o.isWide()) {
                return o.name().toLowerCase(Locale.ROOT);
            }
        }
        return null;
    }

    /**
     * The constant pool count and every entry, in one call.
     *
     * The count is one more than the number of slots used, which is the other half of
     * the pool being one based {[JVMS 4.1@SE25]}. A pool holding two entries says three.
     */
    Cf pool() {
        if (poolWritten) throw new IllegalStateException("the pool is already written");
        poolWritten = true;
        put("constant_pool_count", 2, nextIndex,
            nextIndex + ", so " + (nextIndex - 1) + " slots and the first is 1");
        for (int i = 0; i < entries.size(); i++) {
            raw("#" + entryIndex.get(i), entries.get(i));
            spans.set(spans.size() - 1, new Span(
                "#" + entryIndex.get(i), spans.get(spans.size() - 1).at(),
                entries.get(i).length, entryText.get(i)));
        }
        return this;
    }

    /**
     * A four byte length field to be filled in when the thing it measures is finished.
     *
     * `attribute_length` and `code_length` both say how many bytes come after them, and
     * neither can be written before those bytes exist. A compiler solves this by building
     * the body in a buffer first. Here the field is written as zero, the offset is
     * remembered, and `close` goes back and patches it, which is the same trick every
     * class file writer uses and is worth seeing once.
     */
    Cf openU4(String label) {
        put(label, 4, 0, "filled in by close()");
        pending.push(new int[] { len - 4, 4, spans.size() - 1 });
        return this;
    }

    /** The same, two bytes wide. */
    Cf openU2(String label) {
        put(label, 2, 0, "filled in by close()");
        pending.push(new int[] { len - 2, 2, spans.size() - 1 });
        return this;
    }

    /** Fill in the most recently opened length field with everything written since. */
    Cf close() {
        if (pending.isEmpty()) throw new IllegalStateException("nothing is open to close");
        int[] where = pending.pop();
        int at = where[0], width = where[1], span = where[2];
        long value = len - (at + width);
        for (int i = 0; i < width; i++) buf[at + i] = (byte) (value >> ((width - 1 - i) * 8));
        Span old = spans.get(span);
        spans.set(span, new Span(old.label(), old.at(), old.length(),
            value + " bytes follow, counted by close()"));
        return this;
    }

    /** The class file, as bytes. */
    byte[] bytes() {
        if (!pending.isEmpty()) {
            throw new IllegalStateException(
                pending.size() + " length field(s) opened and not closed, so the file has a "
                + "zero where a length belongs. Every openU4 needs a close");
        }
        return Arrays.copyOf(buf, len);
    }

    int size() { return len; }

    List<Span> spans() { return List.copyOf(spans); }

    /**
     * The field with this label, so an experiment can go back and find one.
     *
     * Labels are matched with the indentation taken off. Indenting a label is how the
     * dump shows what is nested inside what, and a reader who typed four spaces to make
     * the dump readable should not have to count them again here.
     */
    Span field(String label) {
        for (Span s : spans) {
            if (s.label().trim().equals(label.trim())) return s;
        }
        throw new IllegalArgumentException("no field called " + label + " was written");
    }

    /**
     * A copy of the file with one field set to something else, which is how you break it.
     *
     * The interesting question about a class file is not whether a correct one works. It
     * is which of the many things that could be wrong the VM notices, at what point, and
     * with what error class, and the only way to ask that is to write a correct file and
     * then damage exactly one field of it.
     */
    byte[] with(String label, long value) {
        Span s = field(label);
        byte[] copy = bytes();
        for (int i = 0; i < s.length(); i++) {
            copy[s.at() + i] = (byte) (value >> ((s.length() - 1 - i) * 8));
        }
        return copy;
    }

    // -- looking at what you wrote -----------------------------------------------------

    // Wide enough for eight bytes and an ellipsis, and for the longest field name in
    // JVMS 4.1 and 4.7.3, which is exception_table_length under four spaces of indent.
    private static final int HEX_WIDTH = 28;
    private static final int LABEL_WIDTH = 30;

    private String hexOf(Span s) {
        StringBuilder out = new StringBuilder();
        int shown = Math.min(s.length(), 8);
        for (int i = 0; i < shown; i++) {
            out.append(String.format("%02x", buf[s.at() + i]));
            if (i + 1 < shown) out.append(' ');
        }
        if (s.length() > shown) out.append(" ...");
        return out.toString();
    }

    /** Every byte of the file, in order, with what each run of them was for. */
    void dump() {
        String title = len + " bytes, " + spans.size() + " fields";
        if (Ui.html(dumpHtml(title))) return;
        System.out.print(dumpText(title));
    }

    String dumpHtml(String title) {
        StringBuilder rows = new StringBuilder();
        for (Span s : spans) {
            rows.append("<div style=\"font-family:").append(Ui.MONO)
                .append(";font-size:12px;line-height:1.6;white-space:pre\">")
                .append("<span style=\"color:").append(Ui.MUTED).append("\">")
                .append(String.format("%04d", s.at())).append("</span>  ")
                .append("<span style=\"color:").append(Ui.BLUE).append("\">")
                .append(Ui.esc(pad(hexOf(s), HEX_WIDTH))).append("</span>")
                .append(Ui.esc(pad(s.label(), LABEL_WIDTH)))
                .append("<span style=\"color:").append(Ui.MUTED).append("\">")
                .append(Ui.esc(s.note() == null ? "" : s.note()))
                .append("</span></div>");
        }
        return Ui.card(Ui.BLUE, "class file", Ui.line(Ui.prose(title)) + rows);
    }

    String dumpText(String title) {
        StringBuilder out = new StringBuilder(title).append("\n");
        for (Span s : spans) {
            out.append(String.format("%04d  %s%s%s%n", s.at(), pad(hexOf(s), HEX_WIDTH),
                pad(s.label(), LABEL_WIDTH), s.note() == null ? "" : s.note()));
        }
        return out.toString();
    }

    private static String pad(String text, int width) {
        StringBuilder out = new StringBuilder(text);
        while (out.length() < width) out.append(' ');
        return out.append(' ').toString();
    }

    // -- the three opinions --------------------------------------------------------------
    //
    // A class file you wrote by hand is a claim, and there are three parties who can
    // disagree with it. javap parses it with the JDK's own reader and prints what it
    // found. The VM parses it again with different code and different rules, and either
    // links it or refuses. And running it is the only one that tells you the code does
    // what you meant. They fail at different points on purpose: a file javap prints
    // happily can still be refused by the VM, and that gap is where the interesting
    // errors live.

    /**
     * The real javap, on your bytes.
     *
     * Not a reimplementation and not a parser of our own. This writes the file out and
     * hands it to the tool the JDK ships, through ToolProvider, so what comes back is
     * what a reader would get from a terminal.
     */
    static String javap(byte[] bytes, String... options) {
        ToolProvider javap = ToolProvider.findFirst("javap").orElseThrow(
            () -> new UnsupportedOperationException(
                "this runtime has no javap, so the jdk.jdeps module is missing"));
        try {
            Path dir = Files.createTempDirectory("jvx-cf");
            Path file = dir.resolve("Handwritten.class");
            Files.write(file, bytes);
            List<String> args = new ArrayList<>(Arrays.asList(options));
            args.add(file.toString());
            StringWriter out = new StringWriter();
            StringWriter err = new StringWriter();
            javap.run(new PrintWriter(out), new PrintWriter(err), args.toArray(new String[0]));
            Files.deleteIfExists(file);
            Files.deleteIfExists(dir);
            // Merged, because javap reports a file it could not read on stderr and a
            // reader who gets an empty pane has been told nothing.
            return out + err.toString();
        } catch (IOException e) {
            throw new RuntimeException("could not hand the bytes to javap", e);
        }
    }

    /**
     * A loader that will define any bytes once and then never be used again.
     *
     * A fresh one per attempt, which is what makes this a playground: the same class name
     * can be defined again after a fix, and a loader that had already refused it would
     * refuse the corrected one for the wrong reason.
     */
    static class Once extends ClassLoader {
        Once() { super(Cf.class.getClassLoader()); }
        Class<?> take(String name, byte[] bytes) {
            return defineClass(name, bytes, 0, bytes.length);
        }
    }

    /**
     * Define the class here, link it, and hand it back.
     *
     * The two happen at different times and the difference is the whole of chapter 5.
     * Defining parses the file and is where a `ClassFormatError` comes from. Linking runs
     * the verifier and is where a `VerifyError` comes from, and it does not happen until
     * something uses the class, which is why this asks for it explicitly rather than
     * letting a reader believe a file was accepted when it was merely stored.
     */
    static Class<?> load(String binaryName, byte[] bytes) {
        Once loader = new Once();
        Class<?> defined = loader.take(binaryName, bytes);
        try {
            Class.forName(binaryName, true, loader);
        } catch (ClassNotFoundException e) {
            throw new IllegalStateException("defined and then not found: " + binaryName, e);
        }
        return defined;
    }

    /**
     * Run the class in a fresh JVM and hand back everything it printed.
     *
     * Two reasons to leave the kernel. A broken class file that takes the VM down rather
     * than throwing would take the reader's whole session with it, which is why
     * probes/classfile-fuzz runs its mutants in a separate process even though none of
     * its 256 bit flips has managed it. And the flags worth trying on a handwritten class
     * file cannot be changed in a VM that is already running.
     */
    static String launch(String binaryName, byte[] bytes, String... vmArgs) {
        try {
            Path dir = Files.createTempDirectory("jvx-cf");
            Path file = dir.resolve(binaryName.replace('.', '/') + ".class");
            Files.createDirectories(file.getParent());
            Files.write(file, bytes);

            List<String> command = new ArrayList<>();
            command.add(Path.of(System.getProperty("java.home"), "bin", "java").toString());
            command.addAll(Arrays.asList(vmArgs));
            command.add("-cp");
            command.add(dir.toString());
            command.add(binaryName);

            Process p = new ProcessBuilder(command).redirectErrorStream(true).start();
            String out = new String(p.getInputStream().readAllBytes(), StandardCharsets.UTF_8);
            p.waitFor();
            return out;
        } catch (Exception e) {
            throw new RuntimeException("could not run " + binaryName + " in a fresh JVM", e);
        }
    }

    /**
     * What went wrong, in one line, for a file that was meant to be refused.
     *
     * The error class matters more than the message here and is printed first. A reader
     * comparing a rejection against what the specification names is comparing class
     * names, and the message is HotSpot's own wording, which the specification does not
     * govern and which probes/classfile-fuzz found uses none of the words its own rule
     * uses in twelve of twenty seven cases.
     */
    static String refusal(String binaryName, byte[] bytes) {
        try {
            load(binaryName, bytes);
            return "no error: this VM accepted it";
        } catch (Throwable t) {
            return t.getClass().getName() + ": " + t.getMessage();
        }
    }
}

// A prediction gate.
//
// The rule this project runs on is that a reader who has not committed to an answer
// does not really read the reveal. They skim it and come away feeling like they knew.
// Committing to a wrong answer first is what makes the correction stick, so the gate
// makes you write one down and does not show you anything until you have.
//
// There are two renderings and the text one is not a fallback in the apologetic sense.
// It works in a terminal, in a printed transcript and in any notebook with no HTML, and
// every word of it is in the card version too. What the card adds is a shape the eye can
// find on a long page, and one thing the text cannot do: on a page nobody has run, the
// answer sits inside a details element, so a reader scrolling past a reveal has to decide
// to open it rather than have it handed to them. That is the gate working in the one
// environment where it used to be impossible.
//
// A lesson never names Gate. It calls jvx.gate, jvx.answer and jvx.reveal, which is why
// this file could change shape twice without touching a lesson.

class Gate {

    static final Map<String, String> question = new LinkedHashMap<>();
    static final Map<String, String> answered = new LinkedHashMap<>();

    static void ask(String id, String text, String... options) {
        question.put(id, text);
        if (Ui.html(askHtml(id, text, options))) return;

        System.out.println(text);
        System.out.println();
        for (String option : options) {
            System.out.println("    " + option);
        }
        System.out.println();
        System.out.println("Pick one before you run anything else. There is a right answer and");
        System.out.println("the wrong ones are wrong for reasons worth knowing.");
        System.out.println();
        System.out.println("    jvx.answer(\"" + id + "\", \"a\")");
    }

    static void answer(String id, String choice) {
        if (!question.containsKey(id)) {
            String missing = "No gate called " + id + " is open. Run the gate cell above first.";
            if (!Ui.html(Ui.card(Ui.ORANGE, "nothing to answer", Ui.line(Ui.esc(missing))))) {
                System.out.println(missing);
            }
            return;
        }
        String cleaned = choice.trim().toLowerCase();
        answered.put(id, cleaned);
        if (Ui.html(answerHtml(id, cleaned))) return;
        System.out.println("Recorded " + cleaned + " for " + id + ". Now run the next cell.");
    }

    static void reveal(String id, String correct) {
        String expected = correct.trim().toLowerCase();
        String mine = answered.get(id);
        if (Ui.html(revealHtml(id, expected, mine))) return;

        if (mine == null) {
            // Not a refusal. Somebody who hit Run All should not end up staring at a
            // cell that will not talk to them. But it says what was lost, because it
            // was a real thing and not a formality.
            System.out.println("You did not commit to an answer, so this is worth less to you");
            System.out.println("than it would have been. The answer is " + expected + ".");
            return;
        }
        if (mine.equals(expected)) {
            System.out.println("You said " + mine + ", and that is right.");
            System.out.println("Read on anyway. Being right for the wrong reason is common here.");
        } else {
            System.out.println("You said " + mine + ". The answer is " + expected + ".");
            System.out.println("This is the useful outcome. Read what follows carefully.");
        }
    }

    // -- the card version ------------------------------------------------------------
    //
    // These three build markup and put nothing on the screen, which is what makes them
    // testable without a kernel. tools/test_jvx_ui.py runs them in a plain jshell and
    // checks both what they say and that they use only the four things that survive a
    // notebook nobody has run.

    static String askHtml(String id, String text, String[] options) {
        StringBuilder body = new StringBuilder();
        body.append("<div style=\"font-size:15px;line-height:1.5;margin-bottom:10px\">")
            .append(Ui.prose(text))
            .append("</div>");
        for (String option : options) {
            body.append(optionHtml(option));
        }
        body.append(Ui.small(
            "Pick one before you run anything else. There is a right answer and the wrong "
            + "ones are wrong for reasons worth knowing. Put yours in the next cell: "
            + Ui.code("jvx.answer(&quot;" + Ui.esc(id) + "&quot;, &quot;a&quot;)")));
        return Ui.card(Ui.BLUE, "predict", body.toString());
    }

    /**
     * One option, with its letter pulled out into a chip.
     *
     * Lessons write options as "a) 8", so the letter is already there and splitting it
     * out is presentation. An option written some other way is printed whole rather than
     * mangled into the shape this method was hoping for.
     */
    static String optionHtml(String option) {
        String letter = "";
        String rest = option;
        int bracket = option.indexOf(')');
        if (bracket == 1 && Character.isLetter(option.charAt(0))) {
            letter = option.substring(0, 1);
            rest = option.substring(2).trim();
        }
        String chip = letter.isEmpty() ? "" :
            "<span style=\"font-family:" + Ui.MONO + ";font-weight:700;color:" + Ui.BLUE
            + ";margin-right:10px\">" + Ui.esc(letter) + "</span>";
        return "<div style=\"margin:5px 0 5px 4px;line-height:1.5\">"
            + chip + Ui.prose(rest) + "</div>";
    }

    static String answerHtml(String id, String choice) {
        return Ui.card(Ui.MUTED, "recorded",
            Ui.line("You said " + Ui.code(Ui.esc(choice)) + " for "
                + Ui.code(Ui.esc(id)) + ". It is not marked yet. Run the next cell."));
    }

    static String revealHtml(String id, String expected, String mine) {
        if (mine == null) {
            // Nobody ran the answer cell, which is also what a reader of the published
            // page is looking at, since nothing on it has been run at all. Putting the
            // answer behind a details is the whole reason this rendering exists: it is
            // the only interaction that survives an unrun notebook, so it is the only way
            // a gate can still be a gate on a page somebody is only reading.
            return Ui.card(Ui.MUTED, "not answered",
                Ui.line("You did not commit to an answer, so this is worth less to you than "
                    + "it would have been. Write one down and it will mark it.")
                + Ui.details("Show me the answer anyway",
                    Ui.line("The answer is " + Ui.code(Ui.esc(expected)) + "."), false));
        }
        if (mine.equals(expected)) {
            return Ui.card(Ui.GREEN, "right",
                Ui.line("You said " + Ui.code(Ui.esc(mine)) + ", and that is right.")
                + Ui.small("Read on anyway. Being right for the wrong reason is common here."));
        }
        return Ui.card(Ui.ORANGE, "worth having",
            Ui.line("You said " + Ui.code(Ui.esc(mine)) + ". The answer is "
                + Ui.code(Ui.esc(expected)) + ".")
            + Ui.small("This is the useful outcome. Read what follows carefully."));
    }
}

// HeapLens: what one object looks like in memory, byte by byte.
//
// This file draws and measures nothing. It is handed a list of slots that somebody else
// measured and turns them into a picture, which is what makes it testable without a JVM
// to point at: the test can hand it a layout it invented and check the drawing, and
// separately check that the measuring produces the layout it should.
//
// The picture is an SVG inside an img with a data URI, because probes/widgets measured
// that as the only kind of picture that renders in all four places a reader might be,
// including a saved notebook nobody has run. Inside the img the SVG is never sanitized,
// so the drawing can use anything SVG has.

class Lens {

    /**
     * One run of bytes in an object, and what is living there.
     *
     * `kind` is what it is rather than what it looks like: header, field, gap or padding.
     * The difference between a gap and padding matters and is the whole reason a reader
     * is looking at this. A gap is alignment inside the object, put there because the
     * next field could not start where the last one ended. Padding is at the end, put
     * there because the whole object has to be a multiple of the alignment. One is the
     * field order's fault and can be fixed by reordering. The other cannot.
     */
    record Slot(String label, long offset, long width, String kind) {}

    static final int CELL = 34;      // one byte
    static final int ROW = 42;       // one 8 byte word
    static final int GUTTER = 46;    // the offset down the left
    static final int TOP = 28;       // the byte ruler across the top
    static final int PAD = 12;

    static String colour(String kind, int index) {
        if (kind.equals("header")) return "#4c6ef5";
        if (kind.equals("padding")) return "#adb5bd";
        if (kind.equals("gap")) return "#ffa94d";
        // Fields cycle, so two neighbours never share a colour and a reader can see
        // where one stops without reading the label.
        String[] wheel = { "#2f9e44", "#1098ad", "#7048e8", "#e64980", "#f08c00" };
        return wheel[Math.floorMod(index, wheel.length)];
    }

    static String ink(String kind) {
        return kind.equals("padding") ? "#495057" : "#ffffff";
    }

    /** The picture. One row per eight byte word, one rect per slot per row. */
    static String svg(List<Slot> slots, long size) {
        long rows = (size + 7) / 8;
        int width = PAD * 2 + GUTTER + 8 * CELL;
        int height = (int) (TOP + rows * ROW + PAD);

        StringBuilder out = new StringBuilder();
        out.append("<svg xmlns=\"http://www.w3.org/2000/svg\" width=\"").append(width)
           .append("\" height=\"").append(height)
           .append("\" viewBox=\"0 0 ").append(width).append(" ").append(height)
           .append("\" font-family=\"ui-monospace, SFMono-Regular, Menlo, monospace\">");
        out.append("<rect width=\"").append(width).append("\" height=\"").append(height)
           .append("\" fill=\"#ffffff\"/>");

        // The byte ruler. Which byte of the word, not which byte of the object, because
        // the offset down the side already says that and saying it twice is noise.
        for (int b = 0; b < 8; b++) {
            out.append(text(PAD + GUTTER + b * CELL + CELL / 2, TOP - 10, String.valueOf(b),
                11, "#adb5bd", "middle", false));
        }

        for (long row = 0; row < rows; row++) {
            int y = (int) (TOP + row * ROW);
            out.append(text(PAD + GUTTER - 12, y + ROW / 2 + 4, String.valueOf(row * 8),
                12, "#868e96", "end", false));
        }

        int fieldIndex = 0;
        for (Slot slot : slots) {
            int index = slot.kind().equals("field") ? fieldIndex++ : 0;
            long from = slot.offset();
            long to = slot.offset() + slot.width();
            // A slot that crosses a word boundary is drawn once per row it touches, so
            // the picture stays a grid and a long field still reads as one thing.
            for (long start = from; start < to; ) {
                long row = start / 8;
                long end = Math.min(to, (row + 1) * 8);
                int x = (int) (PAD + GUTTER + (start % 8) * CELL);
                int y = (int) (TOP + row * ROW);
                int w = (int) ((end - start) * CELL) - 3;
                out.append("<rect x=\"").append(x + 1).append("\" y=\"").append(y + 3)
                   .append("\" width=\"").append(w).append("\" height=\"").append(ROW - 9)
                   .append("\" rx=\"4\" fill=\"").append(colour(slot.kind(), index))
                   .append(slot.kind().equals("padding")
                       ? "\" fill-opacity=\"0.35\" stroke=\"#adb5bd\" stroke-dasharray=\"3 3\"/>"
                       : "\"/>");
                // A label needs room. Two bytes is not room, and a clipped word is worse
                // than no word, because the offset in the details below is exact anyway.
                if (w >= 62) {
                    out.append(text(x + 1 + w / 2, y + ROW / 2 + 1, slot.label(), 13,
                        ink(slot.kind()), "middle", true));
                }
                start = end;
            }
        }
        out.append("</svg>");
        return out.toString();
    }

    static String text(int x, int y, String body, int size, String fill, String anchor,
                       boolean bold) {
        return "<text x=\"" + x + "\" y=\"" + y + "\" font-size=\"" + size + "\" fill=\"" + fill
            + "\" text-anchor=\"" + anchor + "\""
            + (bold ? " font-weight=\"600\"" : "") + ">" + Ui.esc(body) + "</text>";
    }

    /**
     * What the picture says, for somebody who cannot see it.
     *
     * Not an afterthought and not generated from the same string twice. A screen reader
     * gets this, and so does anyone whose front end blocks images, and it is the same
     * sentence the text version prints in a terminal.
     */
    static String alt(String title, List<Slot> slots, long size) {
        StringBuilder out = new StringBuilder("the layout of " + title + ", " + size + " bytes: ");
        for (int i = 0; i < slots.size(); i++) {
            Slot s = slots.get(i);
            if (i > 0) out.append(", ");
            out.append(s.label()).append(" at ").append(s.offset())
               .append(" for ").append(s.width()).append(s.width() == 1 ? " byte" : " bytes");
        }
        return out.toString();
    }

    /** The card: the picture, then the numbers behind it one click away. */
    static String card(String title, List<Slot> slots, long size, String note) {
        StringBuilder rows = new StringBuilder();
        int fieldIndex = 0;
        for (Slot s : slots) {
            // Counted the same way the picture counts, so the dot beside a name is the
            // colour of the box it points at. Two lists that drift apart are worse than
            // one list, so there is exactly one rule and both of them use it.
            int index = s.kind().equals("field") ? fieldIndex++ : 0;
            // white-space:pre, or the alignment in `row` does nothing: HTML collapses runs
            // of spaces, and a monospace font with collapsed spaces lines nothing up.
            rows.append("<div style=\"margin:3px 0;font-family:" + Ui.MONO
                    + ";font-size:13px;white-space:pre\">")
                .append("<span style=\"display:inline-block;width:9px;height:9px;border-radius:2px;")
                .append("background:").append(colour(s.kind(), index))
                .append(";margin-right:8px\"></span>")
                .append(Ui.esc(row(s)))
                .append("</div>");
        }
        String body =
            Ui.img(svg(slots, size), alt(title, slots, size))
            + Ui.line("<b>" + Ui.esc(title) + "</b> is " + size + " bytes.")
            + Ui.details("The exact offsets", rows.toString(), false)
            + (note.isEmpty() ? "" : Ui.small(Ui.prose(note)));
        return Ui.card(Ui.BLUE, "layout", body);
    }

    /**
     * One slot as a line of text, used by both renderings.
     *
     * One format string rather than two, because the terminal version and the list
     * behind the picture say the same thing and there is no version of this project
     * where it is good for them to drift.
     */
    static String row(Slot s) {
        return String.format("%-18s bytes %2d to %2d  (%d)",
            s.label(), s.offset(), s.offset() + s.width() - 1, s.width());
    }

    /** The same thing for a terminal, where there is no picture to look at. */
    static String text(String title, List<Slot> slots, long size) {
        StringBuilder out = new StringBuilder(title + " is " + size + " bytes\n\n");
        for (Slot s : slots) {
            out.append("  ").append(row(s)).append("\n");
        }
        return out.toString();
    }
}

// jvx is the small helper surface every lesson gets for free. It is deliberately thin.
// Anything it does that a reader could do themselves in three lines, it does in a way
// they can read, and it never hides the tool underneath it. When a lesson wants JOL or
// jcmd or jfr, the lesson calls JOL or jcmd or jfr, because watching the real tool is
// the point and a wrapper would be one more thing to trust.
//
// The class name is lower case. That is not a mistake and not Java style. It is a
// namespace that reads like one at a call site, `jvx.mark(o)`, and every lesson has it.

class jvx {

    static final String PIN = "jdk-27+35";
    static final String BUILT_FROM = "jvx/00-imports.jsh, jvx/05-ui.jsh, jvx/10-markword.jsh, jvx/12-classfile.jsh, jvx/15-gate.jsh, jvx/18-lens.jsh, jvx/20-jvx.jsh";

    // -- reading raw object memory ------------------------------------------------
    //
    // There is no supported API for reading the bytes of an object header. That is not
    // an oversight, it is the whole reason the mark word is an implementation detail:
    // the JVMS does not mandate any internal structure for objects at all (JVMS 2.7),
    // so there is nothing for an API to promise. Two internal doors are open on JDK 27
    // and jvx tries them in this order.
    //
    //   1. jdk.internal.misc.Unsafe, which needs
    //      --add-exports java.base/jdk.internal.misc=ALL-UNNAMED on the command line.
    //      Preferred, because it prints nothing.
    //   2. sun.misc.Unsafe, which needs no flags and prints four lines of terminal
    //      deprecation warning the first time. It still works on JDK 27 and it is on
    //      its way out.
    //
    // Which door opened is not hidden. jvx.markRoute() says, and the banner prints it,
    // because "where did this number come from" is a question a reader is entitled to
    // ask about a number that came from reading memory directly.

    private static Object unsafe;
    private static Method getLongMethod;
    private static Method fieldOffsetMethod;
    private static Method arrayBaseMethod;
    private static Method arrayScaleMethod;
    private static boolean fieldOffsetTakesAField;
    private static String markRoute = "not tried yet";

    private static void openUnsafe() {
        if (getLongMethod != null) return;
        try {
            Class<?> c = Class.forName("jdk.internal.misc.Unsafe");
            unsafe = c.getMethod("getUnsafe").invoke(null);
            getLongMethod = c.getMethod("getLong", Object.class, long.class);
            // This one names the field with a string. The older door wants a
            // reflected Field object instead, which is why the two are not
            // interchangeable and why fieldOffset below has to know which it got.
            fieldOffsetMethod = c.getMethod("objectFieldOffset", Class.class, String.class);
            arrayBaseMethod = c.getMethod("arrayBaseOffset", Class.class);
            arrayScaleMethod = c.getMethod("arrayIndexScale", Class.class);
            fieldOffsetTakesAField = false;
            markRoute = "jdk.internal.misc.Unsafe";
            return;
        } catch (Throwable ignored) {
            // Not exported to us. Fall through to the older door.
        }
        try {
            Class<?> c = Class.forName("sun.misc.Unsafe");
            Field f = c.getDeclaredField("theUnsafe");
            f.setAccessible(true);
            unsafe = f.get(null);
            getLongMethod = c.getMethod("getLong", Object.class, long.class);
            fieldOffsetMethod = c.getMethod("objectFieldOffset", Field.class);
            arrayBaseMethod = c.getMethod("arrayBaseOffset", Class.class);
            arrayScaleMethod = c.getMethod("arrayIndexScale", Class.class);
            fieldOffsetTakesAField = true;
            markRoute = "sun.misc.Unsafe (deprecated for removal, expect a warning)";
            return;
        } catch (Throwable t) {
            markRoute = "neither door opened: " + t;
            throw new UnsupportedOperationException(
                "cannot read object memory on this JVM. Start the kernel with "
                + "--add-exports java.base/jdk.internal.misc=ALL-UNNAMED. " + markRoute);
        }
    }

    static String markRoute() {
        openUnsafe();
        return markRoute;
    }

    /** The eight bytes at offset 0 of an object, exactly as they sit in memory. */
    static long mark(Object o) {
        openUnsafe();
        try {
            return (Long) getLongMethod.invoke(unsafe, o, 0L);
        } catch (Exception e) {
            throw new RuntimeException("reading the mark word failed", e);
        }
    }

    /** The mark word, printed with its fields separated out. */
    static void header(Object o) {
        System.out.print(MarkWord.decode(mark(o)));
    }

    static String hex(long word) {
        return MarkWord.hex(word);
    }

    static long field(Object o, String name) {
        return MarkWord.get(mark(o), name);
    }

    /**
     * The mark word of an object nothing has ever touched.
     *
     * This exists because of a trap that is very easy to fall into and impossible to
     * see. Assigning an object to a top level JShell variable is enough to make
     * something ask for its identity hash, and asking is what makes HotSpot write one
     * into the mark word. So this:
     *
     *     Object o = new Object();
     *
     * hands you an object whose hash field is already filled in, and the "before"
     * measurement you were about to take is gone. The semicolon does not save you.
     * Measured on 27+35-2325: a top level variable reads a nonzero hash with the
     * semicolon on the end, the same allocation stored into a static field of a class
     * declared in the same session reads 0, and one that is never named reads 0.
     *
     * Nothing here can touch the object between allocating it and reading it, because
     * the reference never leaves this method. It is the honest "before".
     */
    static long freshMark() {
        return mark(new Object());
    }

    /** Where every bit position jvx believes in came from. */
    static void provenance() {
        System.out.print(MarkWord.provenance());
    }

    // -- measuring layout ------------------------------------------------------------
    //
    // There is a library for this, JOL, and it is a good one. These four methods are
    // here anyway, because reaching for a download is the difference between a lesson
    // a reader can start in thirty seconds and one they cannot, and because the whole
    // trick fits in a sentence: the header is the thing your first field comes after,
    // so the offset of the first field is the size of the header. Nothing is being
    // hidden here. Read the four methods and you have the technique.

    /** The byte offset of one field within an instance. */
    static long fieldOffset(Class<?> owner, String name) {
        openUnsafe();
        try {
            if (fieldOffsetTakesAField) {
                Field f = owner.getDeclaredField(name);
                return (Long) fieldOffsetMethod.invoke(unsafe, f);
            }
            return (Long) fieldOffsetMethod.invoke(unsafe, owner, name);
        } catch (NoSuchFieldException e) {
            throw new IllegalArgumentException(owner.getName() + " has no field called " + name);
        } catch (Exception e) {
            throw new RuntimeException("could not read the offset of " + name, e);
        }
    }

    /**
     * Where the header stops, in bytes, for instances of this class.
     *
     * This is the offset of the earliest field, which is the same thing. A class with
     * no fields at all has no first field to point at, so it has to say so rather than
     * return a number it does not know.
     */
    static long headerSize(Class<?> type) {
        long earliest = Long.MAX_VALUE;
        for (Class<?> c = type; c != null; c = c.getSuperclass()) {
            for (Field f : c.getDeclaredFields()) {
                if (!Modifier.isStatic(f.getModifiers())) {
                    earliest = Math.min(earliest, fieldOffset(c, f.getName()));
                }
            }
        }
        if (earliest == Long.MAX_VALUE) {
            throw new IllegalArgumentException(
                type.getName() + " has no instance fields, so there is no first field offset "
                + "to measure the header with. Add one field and measure that class instead.");
        }
        return earliest;
    }

    /**
     * How wide a reference is, measured rather than assumed.
     *
     * One slot of an Object[] is one reference, so the array's index scale is the answer.
     * This is 4 with compressed oops and 8 without, and guessing it wrong throws every
     * object size in a lesson off by four bytes per field.
     */
    static long refWidth() {
        openUnsafe();
        try {
            return ((Number) arrayScaleMethod.invoke(unsafe, Object[].class)).longValue();
        } catch (Exception e) {
            throw new RuntimeException("could not read the reference width", e);
        }
    }

    /** How many bytes a field of this type takes up inside an object. */
    static long widthOf(Class<?> type) {
        if (type == long.class || type == double.class) return 8;
        if (type == int.class || type == float.class) return 4;
        if (type == short.class || type == char.class) return 2;
        if (type == byte.class || type == boolean.class) return 1;
        return refWidth();
    }

    /** Where an array's elements start, which is the size of an array header. */
    static long arrayBase(Class<?> arrayType) {
        openUnsafe();
        try {
            return ((Number) arrayBaseMethod.invoke(unsafe, arrayType)).longValue();
        } catch (Exception e) {
            throw new RuntimeException("could not read the array base offset", e);
        }
    }

    /**
     * Wrap a class declaration in a program that measures it, ready for jvx.run.
     *
     * The declaration has to be called Candidate. The launcher class has to come first
     * in the file and has to match the file name, which is how the single file source
     * launcher decides what to run, so the reader's class cannot be the first one.
     */
    static String sizeProbe(String candidateSource) {
        return """
            import jdk.internal.misc.Unsafe;
            import java.lang.reflect.Field;
            import java.lang.reflect.Modifier;

            public class Answer {
                public static void main(String[] args) {
                    Unsafe u = Unsafe.getUnsafe();
                    long first = Long.MAX_VALUE;
                    long end = 0;
                    for (Class<?> c = Candidate.class; c != null; c = c.getSuperclass()) {
                        for (Field f : c.getDeclaredFields()) {
                            if (Modifier.isStatic(f.getModifiers())) continue;
                            long off = u.objectFieldOffset(c, f.getName());
                            first = Math.min(first, off);
                            end = Math.max(end, off + width(u, f.getType()));
                        }
                    }
                    if (first == Long.MAX_VALUE) {
                        System.out.println("Candidate has no instance fields, so give it one");
                        return;
                    }
                    long size = (end + 7) / 8 * 8;
                    System.out.printf("header stops at %d, fields end at %d, object is %d bytes%n",
                        first, end, size);
                }

                static int width(Unsafe u, Class<?> t) {
                    if (t == long.class || t == double.class) return 8;
                    if (t == int.class || t == float.class) return 4;
                    if (t == short.class || t == char.class) return 2;
                    if (t == byte.class || t == boolean.class) return 1;
                    // A reference is as wide as one slot of an Object[], which is where
                    // compressed oops show up as 4 rather than 8.
                    return u.arrayIndexScale(Object[].class);
                }
            }

            """ + candidateSource + "\n";
    }

    // -- HeapLens, the object layout viewer ------------------------------------------
    //
    // Everything here is measured on the VM the reader is on. Nothing is looked up in a
    // table and nothing is assumed from the platform, because the whole lesson is that
    // the answer moved and the books have not caught up. The drawing is in Lens, which
    // is handed the measurements and never takes any.

    /**
     * A class with exactly one field, used as a ruler.
     *
     * A class with no instance fields has no first field to point at, so there is
     * nothing in it to measure the header with. Every non array object on HotSpot has
     * the same header, so measuring it on this one and saying so is better than either
     * refusing to draw `Object` or printing a number from a book.
     */
    private static class Ruler { byte b; }

    static long alignment() {
        String value = flag("ObjectAlignmentInBytes");
        return value == null ? 8L : Long.parseLong(value);
    }

    /** Where the header stops on this VM, for any object that has no fields to ask. */
    static long headerSize() {
        return fieldOffset(Ruler.class, "b");
    }

    /** Every instance field of this class and its superclasses, earliest first. */
    private static List<Field> instanceFields(Class<?> type) {
        List<Field> found = new ArrayList<>();
        for (Class<?> c = type; c != null; c = c.getSuperclass()) {
            for (Field f : c.getDeclaredFields()) {
                if (!Modifier.isStatic(f.getModifiers())) found.add(f);
            }
        }
        found.sort((a, b) -> Long.compare(
            fieldOffset(a.getDeclaringClass(), a.getName()),
            fieldOffset(b.getDeclaringClass(), b.getName())));
        return found;
    }

    /** How many bytes an instance takes, including the padding at the end. */
    static long sizeOf(Class<?> type) {
        long end = headerSize();
        for (Field f : instanceFields(type)) {
            end = Math.max(end,
                fieldOffset(f.getDeclaringClass(), f.getName()) + widthOf(f.getType()));
        }
        long align = alignment();
        return (end + align - 1) / align * align;
    }

    /**
     * The whole object as a list of byte runs, in order, with nothing left out.
     *
     * The gaps are the point. A field that does not start where the last one ended has
     * something in between, and that something is alignment padding the reader did not
     * ask for and is paying for. Naming it in the same list as the fields is what makes
     * it visible.
     */
    static List<Lens.Slot> layout(Class<?> type) {
        List<Lens.Slot> slots = new ArrayList<>();
        long cursor = headerSize();
        slots.add(new Lens.Slot("header", 0, cursor, "header"));
        for (Field f : instanceFields(type)) {
            long at = fieldOffset(f.getDeclaringClass(), f.getName());
            long width = widthOf(f.getType());
            if (at > cursor) {
                slots.add(new Lens.Slot("gap", cursor, at - cursor, "gap"));
            }
            slots.add(new Lens.Slot(
                f.getType().getSimpleName() + " " + f.getName(), at, width, "field"));
            cursor = at + width;
        }
        long size = sizeOf(type);
        if (size > cursor) {
            slots.add(new Lens.Slot("padding", cursor, size - cursor, "padding"));
        }
        return slots;
    }

    /** Draw one object's layout, byte by byte, as measured on this VM. */
    static void lens(Class<?> type) {
        List<Lens.Slot> slots = layout(type);
        long size = sizeOf(type);
        String title = type.getSimpleName();
        String note = "Measured on this VM, where `UseCompactObjectHeaders` is "
            + (flag("UseCompactObjectHeaders") == null ? "not a flag" : flag("UseCompactObjectHeaders"))
            + " and `ObjectAlignmentInBytes` is " + alignment() + ".";
        if (instanceFields(type).isEmpty()) {
            note = title + " has no instance fields, so the header was measured on a class "
                + "that has one. Every non array object on HotSpot has the same header. " + note;
        }
        if (Ui.html(Lens.card(title, slots, size, note))) return;
        System.out.print(Lens.text(title, slots, size));
        System.out.println();
        System.out.println(note.replace("`", ""));
    }

    // -- asking the VM about itself -----------------------------------------------
    //
    // This part needs no internal access at all. HotSpotDiagnosticMXBean is supported
    // API in the jdk.management module and it answers for any flag the VM has,
    // including the origin, which is the part people forget to check. A flag that is
    // true because it is the default and a flag that is true because somebody put it
    // on the command line are different facts, and a lesson that confuses them is
    // teaching a local accident as a general truth.

    private static HotSpotDiagnosticMXBean diagnostic() {
        return ManagementFactory.getPlatformMXBean(HotSpotDiagnosticMXBean.class);
    }

    /** A flag's value, or null when this VM has no such flag. */
    static String flag(String name) {
        try {
            return diagnostic().getVMOption(name).getValue();
        } catch (IllegalArgumentException e) {
            return null;
        }
    }

    /** Where a flag's value came from: default, command line, ergonomic, and so on. */
    static String flagOrigin(String name) {
        try {
            return diagnostic().getVMOption(name).getOrigin().toString();
        } catch (IllegalArgumentException e) {
            return null;
        }
    }

    static boolean on(String name) {
        return "true".equals(flag(name));
    }

    /**
     * A flag with its origin, the way `java -XX:+PrintFlagsFinal` would show it.
     *
     * Formatted into a string and printed once, rather than printf, and that is not a
     * style choice. The kernel turns every write on System.out into its own stream
     * message, and java.util.Formatter writes each padding space separately, so a
     * printf with a %-28s in it arrives at the reader as thirty little pieces and the
     * notebook renders each one on its own line. One string, one println, one line.
     */
    static void flags(String... names) {
        for (String name : names) {
            String value = flag(name);
            if (value == null) {
                System.out.println(String.format(
                    "%-28s %-10s %s", name, "-", "this VM has no such flag"));
            } else {
                System.out.println(String.format(
                    "%-28s %-10s {%s}", name, value, flagOrigin(name).toLowerCase()));
            }
        }
    }

    // -- running something in a different JVM -------------------------------------

    /**
     * Compile and run a single Java file in a fresh JVM, with the flags you give it,
     * and hand back everything it printed.
     *
     * Two quite different jobs need this and it is worth being clear about both.
     *
     * The first is that a lot of what this project teaches is only visible by
     * comparison, and the two things being compared are two JVMs started differently.
     * You cannot turn UseCompactObjectHeaders off in a running VM. The objects are
     * already laid out.
     *
     * The second is that the kernel you are typing into is a JShell, and JShell
     * changes some of what a lesson wants to observe. It wraps every snippet in a
     * synthetic class, which shows up in class histograms and compilation logs, and it
     * touches the objects you assign to variables. A subprocess has none of that,
     * because it is a plain JVM running a plain program.
     *
     * No javac step and no classpath, because a single .java file passed to `java` is
     * compiled in memory and run. That has been standard since JEP 330 in Java 11, and
     * it is why the source in a lesson cell is the whole program rather than the
     * interesting half of one.
     */
    static String run(String className, String source, String... vmArgs) {
        try {
            Path dir = Files.createTempDirectory("jvx");
            Path file = dir.resolve(className + ".java");
            Files.writeString(file, source);

            List<String> command = new ArrayList<>();
            command.add(Path.of(System.getProperty("java.home"), "bin", "java").toString());
            for (String arg : vmArgs) command.add(arg);
            command.add(file.toString());

            // Merged, and on purpose. A VM that refuses a flag says so on stderr, and a
            // reader who gets silence and no output has been told nothing at all.
            Process p = new ProcessBuilder(command).redirectErrorStream(true).start();
            String out = new String(p.getInputStream().readAllBytes());
            p.waitFor();

            Files.deleteIfExists(file);
            Files.deleteIfExists(dir);
            return out;
        } catch (Exception e) {
            throw new RuntimeException("could not run " + className + " in a fresh JVM", e);
        }
    }

    /** The same thing, printed rather than returned, which is what a cell usually wants. */
    static void show(String className, String source, String... vmArgs) {
        System.out.print(run(className, source, vmArgs));
    }

    // -- building a class file by hand ------------------------------------------------
    //
    // Six calls, forwarded to Cf. A reader writes the fields of the file and nothing
    // else; every number they write is one the specification names, and the three ways
    // of finding out whether they got it right are here next to each other on purpose.

    /** A new, empty class file. Write it one field at a time. */
    static Cf cf() {
        return new Cf();
    }

    /** The opcode byte for an instruction name, read off this JDK rather than a table. */
    static int op(String mnemonic) {
        return Cf.opcode(mnemonic);
    }

    /** What the JDK's own disassembler makes of your bytes. */
    static void javap(byte[] bytes, String... options) {
        System.out.print(Cf.javap(bytes, options));
    }

    /** Define your bytes as a class here and link them, so the verifier runs. */
    static Class<?> load(String binaryName, byte[] bytes) {
        return Cf.load(binaryName, bytes);
    }

    /** Run your class in a fresh JVM, with flags, and print what it said. */
    static void launch(String binaryName, byte[] bytes, String... vmArgs) {
        System.out.print(Cf.launch(binaryName, bytes, vmArgs));
    }

    /** The error a file you meant to be refused was refused with, class name first. */
    static String refusal(String binaryName, byte[] bytes) {
        return Cf.refusal(binaryName, bytes);
    }

    // -- prediction gates -----------------------------------------------------------
    //
    // Three calls, forwarded to Gate. A lesson never names Gate, so the day the text
    // version is replaced by a widget, no lesson changes.

    /** Ask a question and stop. Nothing here shows the answer. */
    static void gate(String id, String question, String... options) {
        Gate.ask(id, question, options);
    }

    /** Write your answer down. It is not marked yet. */
    static void answer(String id, String choice) {
        Gate.answer(id, choice);
    }

    /** Mark it. Run this after you have measured, not before. */
    static void reveal(String id, String correct) {
        Gate.reveal(id, correct);
    }

    // -- what am I running on -----------------------------------------------------

    /**
     * Printed by the bootstrap cell of every lesson. It is not decoration. Almost every
     * observation in this project is true of one configuration and false of another, so
     * a reader comparing their output with the page needs to see, on the same screen,
     * which configuration produced theirs.
     */
    static void banner() {
        Runtime.Version v = Runtime.version();
        System.out.println(String.format("java      %s  (%s)",
            v, System.getProperty("java.vm.version")));
        System.out.println(String.format("vm        %s", System.getProperty("java.vm.name")));
        System.out.println(String.format("on        %s %s",
            System.getProperty("os.name"), System.getProperty("os.arch")));
        System.out.println(String.format("lessons pinned to %s", PIN));
        System.out.println();
        // Three flags, not four. UseCompressedClassPointers used to belong in this
        // list and no longer exists on JDK 27, which is exactly why flag() returns
        // null for a missing flag rather than throwing: a banner that dies because
        // the VM moved on is a banner that stops anyone from reading anything.
        flags("UseCompactObjectHeaders", "UseCompressedOops", "ObjectAlignmentInBytes");
        System.out.println();
        System.out.println("mark word read through " + markRoute());
        if (!v.toString().startsWith(PIN.replace("jdk-", "").split("\\+")[0])) {
            System.out.println();
            System.out.println("NOTE: this VM is not the pinned one. Numbers below may differ from the page,");
            System.out.println("      and where they do, this VM is right about this VM and the page is right");
            System.out.println("      about " + PIN + ".");
        }
    }
}

jvx.banner();


## What you are about to write

The file has ten top level fields and then the methods {[JVMS §4.1@SE25]}. In order: the magic number, two version numbers, the constant pool, the access flags, this class, the superclass, the interfaces, the fields, the methods, and the class attributes.

Everything interesting is in the fourth one. The constant pool holds every name, every descriptor and every literal in the file, and the rest of the file is almost entirely indices into it. You cannot write `this_class` until you know which pool entry holds the class name, so the pool is built first even though a reader meets it fourth.

`jvx.cf()` gives you an empty file. Asking it for a pool entry hands back an index and writes nothing yet. Calling `pool()` writes the count and every entry, in one go, at the point in the file where they belong.

In [ ]:
Cf c = jvx.cf();

// Ask for everything the file will need to point at. Each call returns the index that
// entry will have, and identical requests come back with the same index, because a
// constant pool with two copies of "java/lang/Object" in it is a file nobody writes.
int self = c.classEntry("Handwritten");
int parent = c.classEntry("java/lang/Object");
int out = c.fieldref("java/lang/System", "out", "Ljava/io/PrintStream;");
int println = c.methodref("java/io/PrintStream", "println", "(Ljava/lang/String;)V");
int greeting = c.stringEntry("hello from a file I typed");
int mainName = c.utf8("main");
int mainType = c.utf8("([Ljava/lang/String;)V");
int codeName = c.utf8("Code");

System.out.println("this class is #" + self + ", the greeting is #" + greeting);

Two things about that pool are worth stopping on before you write it.

The first index is 1 and not 0 {[JVMS §4.1@SE25]}. Nothing lives at index zero, so a zero where an index belongs can mean "there is no entry here" without being ambiguous, which is how a class with no superclass says so: `java/lang/Object` is the one class in the world whose `super_class` is 0.

The second is that a `CONSTANT_Long` or a `CONSTANT_Double` takes two slots {[JVMS §4.4.5@SE25]}. The specification calls this a poor choice in a footnote, and it is still here. Run the next cell and read the two numbers.

In [ ]:
Cf slots = jvx.cf();
System.out.println("a long is #" + slots.longEntry(42L));
System.out.println("the next entry is #" + slots.utf8("and this is what came after it"));

## The first ten bytes

`0xCAFEBABE`, then a minor version, then a major version {[JVMS §4.1@SE25]}. The major version is what decides which rules apply to the rest of the file, and this JDK writes 71.

The access flags are a bitmask, and the two bits here are `ACC_PUBLIC` and `ACC_SUPER` {[JVMS §4.1@SE25]}. `ACC_SUPER` has meant nothing since Java 8 and every class file still sets it, which is a hint about how much of this format is history rather than design.

In [ ]:
c.u4("magic", ClassFile.MAGIC_NUMBER, "0xcafebabe");
c.u2("minor_version", 0);
c.u2("major_version", ClassFile.latestMajorVersion());
c.pool();
c.u2("access_flags", ClassFile.ACC_PUBLIC | ClassFile.ACC_SUPER, "public super");
c.u2("this_class", self);
c.u2("super_class", parent);
c.u2("interfaces_count", 0);
c.u2("fields_count", 0);
c.u2("methods_count", 1);

## One method, and two lengths you cannot write yet

A method is four fields and then its attributes {[JVMS §4.6@SE25]}, and the code lives in an attribute rather than in the method, which is why an abstract method needs no special case: it has no `Code` attribute and the format has nothing else to say about it.

Both `attribute_length` and `code_length` say how many bytes come after them, so neither can be written until those bytes exist. `openU4` writes a zero and remembers where, `close` goes back and fills in the difference, and that is what every class file writer ever written does at this point.

The instructions are named rather than numbered. `c.op("getstatic")` asks `java.lang.classfile.Opcode` on your JDK for the byte, so the number in your file is the number that JDK's own writer would have used.

In [ ]:
c.u2("  method access_flags", ClassFile.ACC_PUBLIC | ClassFile.ACC_STATIC, "public static");
c.u2("  method name_index", mainName);
c.u2("  method descriptor_index", mainType);
c.u2("  method attributes_count", 1);

c.u2("    attribute_name_index", codeName);
c.openU4("    attribute_length");
c.u2("    max_stack", 2);
c.u2("    max_locals", 1);
c.openU4("    code_length");

c.op("getstatic").u2("      System.out", out);
c.op("ldc").u1("      the greeting", greeting);
c.op("invokevirtual").u2("      println", println);
c.op("return");

c.close();                                  // code_length
c.u2("    exception_table_length", 0);
c.u2("    attributes_count", 0);
c.close();                                  // attribute_length
c.u2("attributes_count", 0);

byte[] bytes = c.bytes();
System.out.println(bytes.length + " bytes");

## Every byte, and what it was for

This is the whole file. There is nothing else in it, and there is no part of it that this page has not made you write.

In [ ]:
c.dump();

## Three opinions

`javap` first, because it is the friendliest. It is the real tool, not a reimplementation: the bytes are written to a temporary file and handed to the disassembler the JDK ships.

In [ ]:
jvx.javap(bytes, "-c", "-p");

Now the VM in this kernel. Defining the class parses the file, and linking it runs the verifier, and the two are different events with different error classes {[JVMS §5.4.1@SE25]}. `jvx.load` does both, so a file that parses and fails to verify fails here rather than looking accepted.

In [ ]:
Class<?> made = jvx.load("Handwritten", bytes);
System.out.println("the VM accepted it: " + made);
System.out.println("its one method is " + made.getDeclaredMethods()[0]);

And a fresh JVM, which is the only one of the three that can tell you the code does what you meant. It runs in its own process, so a file that manages to take a VM down takes that one rather than this notebook.

In [ ]:
jvx.launch("Handwritten", bytes);

## Now break it

A correct class file teaches you the format. A broken one teaches you the VM, because the question worth asking is which of the many things that could be wrong it notices, when, and in what words.

`c.with("max_stack", 0)` hands back a copy of the file with that one field changed and everything else identical. The method below pushes two values, so a `max_stack` of zero is a lie the file tells about itself.

In [ ]:
System.out.println(jvx.refusal("Handwritten", c.with("max_stack", 0)));

That one behaved. A `VerifyError`, and a message that names the stack, which is what {[JVMS §4.9.2@SE25]} and {[JVMS §5.4.1@SE25]} together say should happen: a file that breaks a §4.9 constraint is refused by the verifier.

It behaved because your method has no branches. Add one and the method needs a `StackMapTable` {[JVMS §4.10.1@SE25]}, HotSpot then checks the declared stack size while reading that attribute rather than while verifying, and the same mistake comes back as a `ClassFormatError` saying `bad type array size`. Same rule, same file, different error class, decided by whether the method happens to have a branch in it.

Now run the next cell and read the third line.

In [ ]:
System.out.println("bad magic       " + jvx.refusal("Handwritten", c.with("magic", 0xcafed00dL)));
System.out.println("from the future " + jvx.refusal("Handwritten", c.with("major_version", 99)));
System.out.println("short on locals " + jvx.refusal("Handwritten", c.with("max_locals", 0)));
System.out.println("pool too big    " + jvx.refusal("Handwritten", c.with("constant_pool_count", 400)));

`max_locals` breaks a constraint from the same §4.9 that `max_stack` did, and it comes back as a `ClassFormatError`. HotSpot checks it in the parser, immediately after reading the two numbers and before it has looked at the code {[HOTSPOT src/hotspot/share/classfile/classFileParser.cpp:2290@jdk-27+35]}, and the parser has one error class to throw. The error follows the place rather than the rule, and nothing in the file told you which of the two you would get.

The last line is worth a second look too. Claiming 400 constants when 21 follow produces `Unknown constant tag 0`, which is the truth about the byte the parser reached and says nothing at all about the count that sent it there.

[docs/probes/classfile-fuzz.md](../../docs/probes/classfile-fuzz.md) does this 27 times and counts how often a message uses any of the words its own rule uses. Twelve times out of twenty seven it uses none of them.

## Your turn

The cell below is empty on purpose. Some things worth building, roughly in order of how much they will teach you:

A method that returns a number, so you meet `ireturn` and a descriptor with a return type in it. A second method, which means a second `Code` attribute and a `methods_count` of 2. A field, which is the shape of a method with no code. A constructor, which is a method whose name is `<init>` and which has to call the superclass constructor before it does anything else. A branch, which is where you find out that the moment your code has two paths through it the verifier wants a `StackMapTable` and will not accept the method without one {[JVMS §4.10.1@SE25]}.

That last one is the wall this playground exists to walk you into, and BP-STACKMAP is the blueprint that gets you over it.

In [ ]:
// Yours. jvx.cf() for an empty file, c.dump() to see it, jvx.javap(bytes) to check it.

## Where this goes

Every field you wrote is specified in [BP-CLASSFILE](../../docs/blueprints/BP-CLASSFILE.md), whose section 2 is generated from the same JDK you are running rather than typed, and every instruction you used is in [BP-BYTECODE](../../docs/blueprints/BP-BYTECODE.md) section 3, which counts the opcodes three ways and gets three different numbers.

The helper you have been calling is `jvx/12-classfile.jsh`, and its source is in the bootstrap cell at the top of this page. It is about four hundred lines and it contains no table of opcodes, no table of constant pool tags and no magic number, because all of those are read off your JDK through `java.lang.classfile` at the moment you ask for them.